# 05_reranking: LLM-Based Relevance Reranking

This notebook demonstrates using an LLM as a Cross-Encoder to rerank candidates retrieved from our scraped corpus.


In [1]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv(dotenv_path=r"d:\Study\Prep\.env")

# 1. Scrape and index
url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.content, "html.parser")
corpus = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 100][:6]

embeddings = OpenAIEmbeddings()
db = FAISS.from_texts(corpus, embeddings)
candidates = db.similarity_search("Who introduced retrieval-augmented generation?", k=3)

# 2. LLM Reranker Scoring
scorer_prompt = ChatPromptTemplate.from_template(
    'Score the relevance of this document to the query on a scale of 0.0 to 1.0.\\n'
    'Query: {query}\\nDocument: {doc}\\nOutput JSON format: {{"score": float}}'
)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

scored_candidates = []
for doc in candidates:
    chain = scorer_prompt | model | JsonOutputParser()
    out = chain.invoke({"query": "Who introduced retrieval-augmented generation?", "doc": doc.page_content})
    scored_candidates.append((doc.page_content, out["score"]))

reranked = sorted(scored_candidates, key=lambda x: x[1], reverse=True)
print("Reranked Candidates:")
for text, score in reranked:
    print(f"- Score {score:.4f}: '{text[:100]}...'")


C:\Users\aryan\AppData\Local\Temp\ipykernel_20132\443142530.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Reranked Candidates:
- Score 1.0000: 'The term retrieval-augmented generation (RAG) was introduced in a 2020 paper that described combinin...'
- Score 0.6000: 'Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to ret...'
- Score 0.2000: 'RAG improves LLMs by incorporating information retrieval before generating responses.[3] Unlike LLMs...'


### Output Explanation
- The Bi-Encoder FAISS fetches 3 candidates.
- The LLM Cross-Encoder assigns exact relevance scores based on query matching context, successfully prioritizing the paragraph containing the origin of RAG.
